In [0]:
storage_account_name = "employeecycle123"
storage_account_key = "l4gsv1cop8i60nKLp5MMy7brFwyjR2QeC4D+vHcl6M6hOfMXcs25q8jUCcr8v0zTePg2vCgRgICC+AStizgj+Q=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

In [0]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("wasbs://bronze@employeecycle123.blob.core.windows.net/day_1/")

In [0]:
df.show(5)

+------+-------+----------+------+----------+------+
|emp_id|   name|department|salary| join_date|status|
+------+-------+----------+------+----------+------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|
+------+-------+----------+------+----------+------+
only showing top 5 rows


In [0]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- status: string (nullable = true)



In [0]:
print("Total Records:", df.count())

Total Records: 200


In [0]:
from pyspark.sql.functions import col
df.select(
    [col(c).isNull().alias(c) for c in df.columns]
).show()

+------+-----+----------+------+---------+------+
|emp_id| name|department|salary|join_date|status|
+------+-----+----------+------+---------+------+
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|
| false|false|     false| false|    false| false|


In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .save("dbfs:/EmployeeLifecycleProject/bronze/day_1")

In [0]:
bronze_df = spark.read.format("delta").load(
    "dbfs:/EmployeeLifecycleProject/bronze/day_1"
)
bronze_df.show(5)

+------+-------+----------+------+----------+------+
|emp_id|   name|department|salary| join_date|status|
+------+-------+----------+------+----------+------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|
+------+-------+----------+------+----------+------+
only showing top 5 rows


In [0]:
days = ["day_2", "day_3", "day_4", "day_5"]
for day in days:

    print(f"Processing {day}")

    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(f"wasbs://bronze@{storage_account_name}.blob.core.windows.net/{day}/")

    print(df.count())

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"dbfs:/EmployeeLifecycleProject/bronze/{day}")

    print(f"{day} completed")

Processing day_2
205
day_2 completed
Processing day_3
210
day_3 completed
Processing day_4
215
day_4 completed
Processing day_5
220
day_5 completed


In [0]:
for day in ["day_1", "day_2", "day_3", "day_4", "day_5"]:

    df = spark.read.format("delta").load(
        f"dbfs:/EmployeeLifecycleProject/bronze/{day}"
    )

    print(f"{day}: {df.count()} records")

day_1: 200 records
day_2: 205 records
day_3: 210 records
day_4: 215 records
day_5: 220 records
